# SafeRoads Pothole Detection Model Training (YOLOv8s)
This notebook fine-tunes a **YOLOv8s** (Small) model on the **IVCNZ Real Pothole Dataset** (1,243 annotated road images).

### Instructions:
1. Run Section 1 to check GPU availability.
2. Run Section 2 to install dependencies (`ultralytics`).
3. Run Section 3 to download and extract the dataset.
4. Run Section 4 to start 30-50 epoch training.
5. Download `runs/pothole_yolov8s/weights/best.pt` and place it in your local `ai-service/models/best.pt` directory.

In [ ]:
# Section 1: Check GPU availability
!nvidia-smi

In [ ]:
# Section 2: Install dependencies
!pip install -q ultralytics pyyaml matplotlib torch torchvision

In [ ]:
# Section 3: Download & Prepare IVCNZ Dataset
import os
import urllib.request
import zipfile
import random
import shutil
from pathlib import Path

DATASET_URL = "https://github.com/jaygala24/pothole-detection/releases/download/v1.0.0/Pothole.Dataset.IVCNZ.zip"
dataset_dir = Path("./dataset")
train_img = dataset_dir / "images" / "train"
val_img = dataset_dir / "images" / "val"
train_lbl = dataset_dir / "labels" / "train"
val_lbl = dataset_dir / "labels" / "val"

for d in [train_img, val_img, train_lbl, val_lbl]:
    d.mkdir(parents=True, exist_ok=True)

yaml_content = f"""path: {dataset_dir.resolve().as_posix()}
train: images/train
val: images/val

names:
  0: pothole
"""
with open(dataset_dir / "data.yaml", "w") as f:
    f.write(yaml_content)

zip_path = Path("./pothole_dataset.zip")
extract_dir = Path("./temp_extract")

print("Downloading IVCNZ dataset...")
urllib.request.urlretrieve(DATASET_URL, zip_path)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

raw_images = list(extract_dir.rglob("*.jpg")) + list(extract_dir.rglob("*.png"))
pairs = [(img, img.with_suffix(".txt")) for img in raw_images if img.with_suffix(".txt").exists()]
print(f"Found {len(pairs)} image-label pairs.")

random.seed(42)
random.shuffle(pairs)
num_val = int(len(pairs) * 0.2)
val_pairs, train_pairs = pairs[:num_val], pairs[num_val:]

for img, txt in train_pairs:
    shutil.copy(img, train_img / img.name)
    shutil.copy(txt, train_lbl / txt.name)

for img, txt in val_pairs:
    shutil.copy(img, val_img / img.name)
    shutil.copy(txt, val_lbl / txt.name)

shutil.rmtree(extract_dir, ignore_errors=True)
zip_path.unlink(missing_ok=True)
print(f"Dataset prepared: {len(train_pairs)} train, {len(val_pairs)} val.")

In [ ]:
# Section 4: Train YOLOv8s Model (30-50 Epochs)
from ultralytics import YOLO

# Load pre-trained YOLOv8s base weights
model = YOLO('yolov8s.pt')

results = model.train(
    data=str(dataset_dir / 'data.yaml'),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,
    name='pothole_yolov8s',
    project='runs',
    mosaic=1.0,
    mixup=0.1,
    degrees=10.0,
    scale=0.5,
    fliplr=0.5,
    save=True,
    plots=True
)

In [ ]:
# Section 5: Validate Model & Display Metrics
metrics = model.val()
print(f"Precision: {metrics.results_dict['metrics/precision(B)']*100:.2f}%")
print(f"Recall:    {metrics.results_dict['metrics/recall(B)']*100:.2f}%")
print(f"mAP50:     {metrics.results_dict['metrics/mAP50(B)']*100:.2f}%")
print(f"mAP50-95:  {metrics.results_dict['metrics/mAP50-95(B)']*100:.2f}%")

In [ ]:
# Section 6: Download best.pt weights
from google.colab import files
best_weights = Path("runs/pothole_yolov8s/weights/best.pt")
if best_weights.exists():
    files.download(str(best_weights))
else:
    print("Weights file not found at:", best_weights)